In [20]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm
GENE_PANEL = ["ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
              "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"]

sborgato@ulisse:/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_timecourse$ ls *_dir/filtered_annotated_saver_ribomito_*_log2_pc1_cpm.csv
CRC0322_12d_EGF_dir/filtered_annotated_saver_ribomito_CRC0322_12d_EGF_log2_pc1_cpm.csv
CRC0322_12d_NOEGF_dir/filtered_annotated_saver_ribomito_CRC0322_12d_NOEGF_log2_pc1_cpm.csv
CRC0322_3d_CTX_dir/filtered_annotated_saver_ribomito_CRC0322_3d_CTX_log2_pc1_cpm.csv
CRC0322_4d_EGF_dir/filtered_annotated_saver_ribomito_CRC0322_4d_EGF_log2_pc1_cpm.csv
CRC0322_4d_NOEGF_dir/filtered_annotated_saver_ribomito_CRC0322_4d_NOEGF_log2_pc1_cpm.csv
CRC0322_7d_CTX_dir/filtered_annotated_saver_ribomito_CRC0322_7d_CTX_log2_pc1_cpm.csv
CRC0322_8d_EGF_dir/filtered_annotated_saver_ribomito_CRC0322_8d_EGF_log2_pc1_cpm.csv
CRC0322_8d_NOEGF_dir/filtered_annotated_saver_ribomito_CRC0322_8d_NOEGF_log2_pc1_cpm.csv


In [21]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC0322']
tratt_cercato=['12d_EGF','12d_NOEGF','3d_CTX','4d_EGF','4d_NOEGF','7d_CTX','8d_EGF','8d_NOEGF']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]+'_'+str.split(file,sep='_')[6]
    print(trattamento)
    if sample_name in train_id and trattamento in tratt_cercato:
        print(file)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data['trattamento']=trattamento
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

cetux_1
NT_1
cetux_2
NT_2
CTX72h_1
NT72h_1
CTX72h_1
NT72h_1
CTX1w_1
NT1w_1
CTX1w_2
NT1w_2
CTX72h_1
NT72h_1
cetux_1
NT_1
cetux_1
NT_1
cetux_1
NT_1
12d_EGF
filtered_annotated_saver_ribomito_CRC0322_12d_EGF_log2_pc1_cpm.csv
12d_NOEGF
filtered_annotated_saver_ribomito_CRC0322_12d_NOEGF_log2_pc1_cpm.csv
3d_CTX
filtered_annotated_saver_ribomito_CRC0322_3d_CTX_log2_pc1_cpm.csv
4d_EGF
filtered_annotated_saver_ribomito_CRC0322_4d_EGF_log2_pc1_cpm.csv
4d_NOEGF
filtered_annotated_saver_ribomito_CRC0322_4d_NOEGF_log2_pc1_cpm.csv
7d_CTX
filtered_annotated_saver_ribomito_CRC0322_7d_CTX_log2_pc1_cpm.csv
8d_EGF
filtered_annotated_saver_ribomito_CRC0322_8d_EGF_log2_pc1_cpm.csv
8d_NOEGF
filtered_annotated_saver_ribomito_CRC0322_8d_NOEGF_log2_pc1_cpm.csv


In [22]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat/filtered_annotated_saver_ribomito_CRC0322_cetux_1_log2_pc1_cpm.csv'
reff=pd.read_csv(data_path,header=0,index_col=0)
reff=reff.T
reff['sample']='CRC0322'
reff['cell_id']=reff.index
reff['trattamento']='cetux'
reff.reset_index(drop=True,inplace=True)
train=pd.concat([train,reff])

In [23]:
train.head()

,ENSG00000000003:TSPAN6,ENSG00000000005:TNMD,ENSG00000000419:DPM1,ENSG00000000457:SCYL3,ENSG00000000460:C1orf112,ENSG00000000938:FGR,ENSG00000000971:CFH,ENSG00000001036:FUCA2,ENSG00000001084:GCLC,ENSG00000001167:NFYA,...,ENSG00000282872:C1orf232,ENSG00000282881:TMEM275,ENSG00000282936:AC004706.3,ENSG00000282988:AL031777.2,ENSG00000283039:KLF18,ENSG00000283071:LBHD2,ENSG00000283093:CENPVL2,sample,cell_id,trattamento
0,8.376196,1.748084,6.902568,3.542765,2.395512,0.0,0.0,6.945957,5.911844,3.769001,...,0.105813,0.0,0.465195,0.542609,0.0,0.0,0.0,CRC0322,AAACGAAGTTCTCTCG.1,12d_EGF
1,8.197334,0.774368,7.397249,3.728476,2.578476,0.0,0.0,7.518196,6.136617,3.960144,...,0.122746,0.0,0.530096,2.556866,0.0,0.0,0.0,CRC0322,AAACGAATCGTGTCAA.1,12d_EGF
2,8.587400,1.707172,6.724211,3.494276,2.365375,0.0,0.0,6.983764,6.155764,3.717712,...,0.105016,0.0,0.380759,1.803938,0.0,0.0,0.0,CRC0322,AAACGCTGTTGCATGT.1,12d_EGF
3,7.462649,2.812242,6.647812,3.661790,2.509400,0.0,0.0,6.906755,5.309100,3.922172,...,0.108666,0.0,0.476281,1.402090,0.0,0.0,0.0,CRC0322,AAAGAACAGCATCTTG.1,12d_EGF
4,8.594062,1.184658,7.039707,3.582610,2.447800,0.0,0.0,7.361362,5.941859,3.812419,...,0.110455,0.0,0.398650,1.760031,0.0,0.0,0.0,CRC0322,AAAGAACGTTAAACAG.1,12d_EGF


In [25]:
from func_NT_cetux import *
n_hvg=1500
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample",'trattamento'), sep=":", on_duplicate="first")

adata_full = make_anndata_from_df(df_clean, set_raw=True)


In [26]:
n_hvg = 1500
batch_key: str = DEFAULT_BATCH_KEY
def select_hvg_cell_ranger_(
    adata,
    n_top_genes=1000,
    batch_key=None,    
    subset=True, n_bins=50,
):
    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat",
        n_top_genes=n_top_genes,
        batch_key=batch_key,  
        subset=subset
    )

select_hvg_cell_ranger_(adata_full, n_top_genes=n_hvg, batch_key=None, subset=True)




In [27]:
adata_full

AnnData object with n_obs × n_vars = 35631 × 1500
    obs: 'cell_id', 'sample', 'trattamento', 'uid'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg'

In [28]:
is_ref = (
    (adata_full.obs["sample"] == "CRC0322") &
    (adata_full.obs["trattamento"] == "cetux")
)

adata_ref   = adata_full[is_ref].copy()



In [29]:
### adataref_CRC0322_cetux scale pca umap
scale_and_pca(adata_ref, n_comps=42, max_value=10)
sc.pp.neighbors(adata_ref, n_pcs=20)
sc.tl.umap(adata_ref)


In [30]:
adata_full.obs["sample_tratt"] = (
    adata_full.obs["sample"].astype(str) + "_" +
    adata_full.obs["trattamento"].astype(str)
)

groups = adata_full.obs["sample_tratt"].unique()

In [31]:
ref_id='CRC0322_cetux'
ingested_list = []

for g in groups:
    if g == ref_id:
        continue  # salta il reference
    
    print("→ Ingest di:", g)
    
    # Estraggo tutte le cellule di quella combinazione
    adata_q = adata_full[adata_full.obs["sample_tratt"] == g].copy()
    
    # Allineo i geni al reference (solo HVG del reference!)
    adata_q = adata_q[:, adata_ref.var_names].copy()
    
    # Ingest
    sc.tl.ingest(adata_q, adata_ref, embedding_method="umap")
    
    ingested_list.append(adata_q)


→ Ingest di: CRC0322_12d_EGF


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


→ Ingest di: CRC0322_12d_NOEGF


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


→ Ingest di: CRC0322_3d_CTX


...

/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


→ Ingest di: CRC0322_8d_EGF


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


→ Ingest di: CRC0322_8d_NOEGF


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [32]:
adata_ingested = adata_ref.concatenate(*ingested_list)


/tmp/ipykernel_526718/2191290189.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_ingested = adata_ref.concatenate(*ingested_list)


In [33]:
#sc.tl.ingest(adata_query, adata_ref, embedding_method="umap")
data_obs = pd.DataFrame(adata_ingested.obs)

#aggiungo umap alle obs perche di default sta nel obsm
data_obs["umap1"] = adata_ingested.obsm["X_umap"][:, 0]
data_obs["umap2"] = adata_ingested.obsm["X_umap"][:, 1]

In [34]:
path_to_save='/mnt/cold2/snaketree/prj/PPH/local/share/data/umap_ingested/'
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]+'_'+str.split(file,sep='_')[6]
    trattamento_sample=sample_name+'_'+trattamento
    if trattamento_sample=='CRC0322_NT':
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_3000_fake_umap_ingested.csv'
    else:
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_umap_ingested.csv'
    if sample_name in train_id and trattamento in tratt_cercato:
        print(trattamento_sample)
        print(nome_file)
        tmp=data_obs[(data_obs['sample'] == sample_name) & (data_obs['trattamento'] == trattamento)]
        cols=['cell_id','umap1','umap2']
        print(tmp.head())
        tmp=tmp.loc[:,cols].reset_index(drop=True)
        file=path_to_save+nome_file
        tmp.to_csv(file)

CRC0322_12d_EGF
CRC0322_12d_EGF_umap_ingested.csv
                                                        cell_id   sample  \
uid                                                                        
CRC032212d_EGF|AAACGAAGTTCTCTCG.1|12d_EGF-1  AAACGAAGTTCTCTCG.1  CRC0322   
CRC032212d_EGF|AAACGAATCGTGTCAA.1|12d_EGF-1  AAACGAATCGTGTCAA.1  CRC0322   
CRC032212d_EGF|AAACGCTGTTGCATGT.1|12d_EGF-1  AAACGCTGTTGCATGT.1  CRC0322   
CRC032212d_EGF|AAAGAACAGCATCTTG.1|12d_EGF-1  AAAGAACAGCATCTTG.1  CRC0322   
CRC032212d_EGF|AAAGAACGTTAAACAG.1|12d_EGF-1  AAAGAACGTTAAACAG.1  CRC0322   

                                            trattamento  \
uid                                                       
CRC032212d_EGF|AAACGAAGTTCTCTCG.1|12d_EGF-1     12d_EGF   
CRC032212d_EGF|AAACGAATCGTGTCAA.1|12d_EGF-1     12d_EGF   
CRC032212d_EGF|AAACGCTGTTGCATGT.1|12d_EGF-1     12d_EGF   
CRC032212d_EGF|AAAGAACAGCATCTTG.1|12d_EGF-1     12d_EGF   
CRC032212d_EGF|AAAGAACGTTAAACAG.1|12d_EGF-1     12d_EGF   

   

In [88]:
col_to_keep=['cell_id','umap1','umap2']
for sample in data_obs['sample'].unique():
    for trattamento in data_obs['trattamento'].unique():
        print(sample)
        print(trattamento)
        tmp = data_obs[(data_obs['sample'] == sample) & (data_obs['trattamento'] == trattamento)]
        tmp=tmp.loc[:,col_to_keep]
        tmp.reset_index(drop=True,inplace=True)
        if not tmp.empty:
                    print(tmp.head())
    

CRC0322
cetux
              cell_id      umap1     umap2
0  AAACCCACACACCAGC.1   1.453070 -8.707530
1  AAACCCACACTGGCCA.1   2.185792 -8.242378
2  AAACCCAGTGTCTTCC.1  13.541674 -2.716104
3  AAACGAATCGAAGCCC.1   0.624740 -6.635291
4  AAACGCTGTCTGCATA.1   0.209044 -5.927050
CRC0322
NT
              cell_id      umap1      umap2
0  CATGGATGTGTGTGGA.1  11.011012  10.406802
1  TAGAGTCAGTACTGTC.1   9.775522  13.148332
2  AGGTAGGAGATGCAGC.1  10.108926  11.344438
3  CATGCCTAGGTGCCTC.1   9.303963  12.945077
4  TGAATGCAGAACCGCA.1  15.845017   7.610606
CRC0322
CTX72h
CRC0322
NT72h
CRC0327
cetux
              cell_id      umap1     umap2
0  AAACCCAAGTTGGAGC.1  10.836754 -7.062515
1  AAACCCACAACCACGC.1   9.432808 -7.299856
2  AAACCCACAGGACTAG.1   9.406543 -5.205430
3  AAACCCACATGTGGTT.1  10.610456 -7.210672
4  AAACCCATCGCCTTGT.1  11.825884 -6.379311
CRC0327
NT
              cell_id      umap1      umap2
0  AAACCCAAGGACAGCT.1  11.349401  14.115401
1  AAACCCAAGGTCACTT.1  11.464429  15.343482
2  AAACCC

In [97]:
path_to_save='/mnt/cold2/snaketree/prj/PPH/local/share/data/umap_ingested/'
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    trattamento_sample=sample_name+'_'+trattamento
    if trattamento_sample=='CRC0322_NT':
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_3000_umap_ingested.csv'
    else:
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_umap_ingested.csv'
    if sample_name in train_id and trattamento in tratt_cercato:
        print(trattamento_sample)
        print(nome_file)
        tmp=data_obs[(data_obs['sample'] == sample_name) & (data_obs['trattamento'] == trattamento)]
        cols=['cell_id','umap1','umap2']
        print(tmp.head())
        tmp=tmp.loc[:,cols].reset_index(drop=True)
        file=path_to_save+nome_file
        tmp.to_csv(file)

CRC0322_cetux
CRC0322_cetux_1_umap_ingested.csv
                                                  cell_id   sample  \
uid                                                                  
CRC0322cetux|AAACCCACACACCAGC.1|cetux  AAACCCACACACCAGC.1  CRC0322   
CRC0322cetux|AAACCCACACTGGCCA.1|cetux  AAACCCACACTGGCCA.1  CRC0322   
CRC0322cetux|AAACCCAGTGTCTTCC.1|cetux  AAACCCAGTGTCTTCC.1  CRC0322   
CRC0322cetux|AAACGAATCGAAGCCC.1|cetux  AAACGAATCGAAGCCC.1  CRC0322   
CRC0322cetux|AAACGCTGTCTGCATA.1|cetux  AAACGCTGTCTGCATA.1  CRC0322   

                                      trattamento  \
uid                                                 
CRC0322cetux|AAACCCACACACCAGC.1|cetux       cetux   
CRC0322cetux|AAACCCACACTGGCCA.1|cetux       cetux   
CRC0322cetux|AAACCCAGTGTCTTCC.1|cetux       cetux   
CRC0322cetux|AAACGAATCGAAGCCC.1|cetux       cetux   
CRC0322cetux|AAACGCTGTCTGCATA.1|cetux       cetux   

                                                                         uid  \
uid      